In [2]:
!pip install transformers==4.25.0
!pip install tokenizers
!pip install sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 13.2 MB/s eta 0:00:0000:0100:01
Reason for being yanked: Version was not properly set
  Attempting uninstall: transformers
    Found existing installation: transformers 4.20.1
    Uninstalling transformers-4.20.1:
      Successfully uninstalled transformers-4.20.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
allennlp 2.10.1 requires transformers<4.21,>=4.1, but you have transformers 4.25.0 which is incompatible.


In [3]:
!pip install pytorch-lightning

In [4]:
# mostly pl is used while doing complex model training
import pytorch_lightning as pl
print(pl.__version__)

1.9.0


In [5]:
import argparse
import glob
from string import punctuation
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import textwrap

from pytorch_lightning.callbacks import ModelCheckpoint

'''
optimizer - AdamW
T5 Conditional Generator in which we'll give conditions
T5 tokenizer because it is fast
training the model without a learning rate
'''
from transformers import (
    AdamW,
    T5ForConditionalGeneration,T5Config,
    T5Tokenizer,
)

In [6]:
pip install clean-text

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.4/175.4 kB 1.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.7 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.7.0-py3-none-any.whl size=171046 sha256=b602ac29f1a8a33fb08d717e7c2187d9fa55c2d073f27be83682c2ce93f670ef
  Stored in directory: /root/.cache/pip/wheels/8a/4e/b6/57b01db010d17ef6ea9b40300af725ef3e210cb1acfb7ac8b6
Successfully built emoji
  Attempting uninstall: emoji
    Found existing installation: emoji 2.2.0
    Uninstalling emoji-2.2.0:
      Successfully uninstalled emoji-2.2.0
Note: you may need to restart the kernel to use updated packages.


In [7]:
from cleantext import clean
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [8]:
pl.seed_everything(32)  

32

In [9]:
! pip install datasets

In [10]:
#ADDITIONALS
!pip install -q wordcloud
import wordcloud

import nltk

In [11]:
# ADDITIONALS
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from datasets import load_dataset                                          
data = load_dataset("squad", split="train[:5000]")

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset squad downloaded and prepared to /root/.cache/huggingface/datasets/squad/plain_text/1.0.0/d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453. Subsequent calls will reuse this data.


In [13]:
print(data)

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 5000
})


In [14]:
f'<start>{data[0]["question"]}<con>{data[0]["context"]}<ans>{data[0]["answers"]["text"][0]}<end>'

'<start>To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<con>Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.<ans>Saint Bernadette Soubirous<end>'

In [15]:
 nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Cleaning

In [16]:
def cleaning(text,punct):
    cleaned_text = clean(text,
        fix_unicode=False,               
        to_ascii=False,                  
        lower=True,                     
        no_line_breaks=False,           
        no_urls=False,                  
        no_emails=False,                
        no_phone_numbers=False,         
        no_numbers=False,               
        no_digits=False,                
        no_currency_symbols=False,      
        no_punct=punct,                 
        lang="en"                       
    )
    
    tokens = word_tokenize(cleaned_text)

    filtered_sentence = [w for w in tokens if not w in stopwords.words('english')]
    cleaned_text_0 = ' '.join(filtered_sentence)
    return cleaned_text_0

print(data[0]['context'],'\n')
cleaning(data[0]['context'],True)

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary. 



'architecturally school catholic character atop main buildings gold dome golden statue virgin mary immediately front main building facing copper statue christ arms upraised legend venite ad omnes next main building basilica sacred heart immediately behind basilica grotto marian place prayer reflection replica grotto lourdes france virgin mary reputedly appeared saint bernadette soubirous 1858 end main drive direct line connects 3 statues gold dome simple modern stone statue mary'

In [17]:
que = []
con = []
ans = []
for i in data:
    que.append(cleaning(i['question'],True)) #was false before
    con.append(cleaning(i['context'],True))
    ans.append(cleaning(i['answers']['text'],True))
    
print(len(que),len(con),len(ans))

5000 5000 5000


In [18]:
que[4],con[4],ans[4]

('sits top main building notre dame',
 'architecturally school catholic character atop main buildings gold dome golden statue virgin mary immediately front main building facing copper statue christ arms upraised legend venite ad omnes next main building basilica sacred heart immediately behind basilica grotto marian place prayer reflection replica grotto lourdes france virgin mary reputedly appeared saint bernadette soubirous 1858 end main drive direct line connects 3 statues gold dome simple modern stone statue mary',
 'golden statue virgin mary')

In [19]:
#training dataset
df = pd.DataFrame()
df['question'] = que
df['context'] = con
df['answers'] = ans
df.head()


,question,context,answers
0,virgin mary allegedly appear 1858 lourdes france,architecturally school catholic character atop...,saint bernadette soubirous
1,front notre dame main building,architecturally school catholic character atop...,copper statue christ
2,basilica sacred heart notre dame beside structure,architecturally school catholic character atop...,main building
3,grotto notre dame,architecturally school catholic character atop...,marian place prayer reflection
4,sits top main building notre dame,architecturally school catholic character atop...,golden statue virgin mary


In [20]:
print (type(df))

<class 'pandas.core.frame.DataFrame'>


# Function to Create a pandas dataframes of questions and answers

In [21]:
df.shape

(5000, 3)

In [22]:
df = df.applymap(str)

In [23]:
df.head()

,question,context,answers
0,virgin mary allegedly appear 1858 lourdes france,architecturally school catholic character atop...,saint bernadette soubirous
1,front notre dame main building,architecturally school catholic character atop...,copper statue christ
2,basilica sacred heart notre dame beside structure,architecturally school catholic character atop...,main building
3,grotto notre dame,architecturally school catholic character atop...,marian place prayer reflection
4,sits top main building notre dame,architecturally school catholic character atop...,golden statue virgin mary


# Tokenization

In [24]:
MODEL_NAME ='t5-base'

In [25]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

Downloading:   0%|          | 0.00/792k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

/opt/conda/lib/python3.7/site-packages/transformers/models/t5/tokenization_t5.py:173: FutureWarning: This tokenizer was incorrectly instantiated with a model max length of 512 which will be corrected in Transformers v5.
For now, this behavior is kept to avoid breaking backwards compatibility when padding/encoding with `truncation is True`.
- Be aware that you SHOULD NOT rely on t5-base automatically truncating your input to 512 when padding/encoding.
- If you want to encode/pad to sequences longer than 512 you can either instantiate this tokenizer with `model_max_length` or pass `max_length` when encoding/padding.
- To avoid this warning, please instantiate this tokenizer with `model_max_length` set to your preferred value.
  FutureWarning,


## To create dataset

In [26]:
class SquadQADataset(Dataset):
  def __init__(
      self,
      data:pd.DataFrame,
      tokenizer:T5Tokenizer,
      source_max_token_len: int = 396,
      target_max_token_len: int = 62,
#32
      ):
    
    self.data =  data
    self.tokenizer =  tokenizer
    self.source_max_token_len =  source_max_token_len
    self.target_max_token_len =  target_max_token_len


  def __len__(self):
    return len(self.data)

  def __getitem__(self, index: int):
    data_row = self.data.iloc[index]

    source_encoding = tokenizer(
      data_row['question'],
      data_row['context'],
      max_length=self.source_max_token_len,
      padding='max_length',
      truncation= True,                    
      return_attention_mask=True,
      add_special_tokens=True,
      return_tensors="pt"
      ) 
    
    target_encoding = tokenizer(
      data_row['answers'],
      max_length=self.target_max_token_len,
      padding='max_length',
      truncation=True,
      return_attention_mask=True,
      add_special_tokens=True,
      return_tensors="pt"
      )
    
    labels = target_encoding['input_ids']
    labels[labels==0] = -100

    return dict(
        question=data_row['question'],
        context=data_row['context'],
        answer_text=data_row['answers'],
        input_ids=source_encoding["input_ids"].flatten(),
        attention_mask=source_encoding['attention_mask'].flatten(),
        labels=labels.flatten()
    )




In [27]:
sample_dataset = SquadQADataset(df, tokenizer)


In [28]:
sample_dataset
# print("Question: ", data['question'][1])

## Splitting into train and validation sets

In [30]:
#train_df, v = train_test_split(df, test_size=0.05)
#df_train,df_test = train_test_split(df, test_size=0.15)

In [31]:
df = df.sample(frac=1,random_state=32).reset_index(drop=True)
train_df = df[0:3800]   #changed as traiing data had validation data included in it
test_df = df[4000:5000].reset_index(drop=True) #added reset

In [32]:
test_df.head()

,question,context,answers
0,one feature could implemented ocarina time,2003 nintendo announced new legend zelda game ...,horseback combat
1,2001 citys mayor picked kill mockingbird favor...,2001 lee inducted alabama academy honor year c...,chicago
2,chopins fathers first name,fryderyks father nicolas chopin frenchman lorr...,nicolas
3,chopins works shows folk aspects,chopins mazurkas waltzes straightforward terna...,mazurkas
4,us share satellite images,may 16 rescue groups south korea japan singapo...,chinese authorities


In [33]:
train_df.head()

,question,context,answers
0,publication alyssa rosenberg write,christopher orr writing atlantic also criticis...,washington post
1,frédéric trouble playing result growing illness,3 december chopin complained bad health incomp...,piano
2,name beyoncés first solo album,beyoncé giselle knowlescarter biːˈjɒnseɪ beeyo...,dangerously love
3,long birth child performed,january 7 2012 beyoncé gave birth first child ...,five months
4,many students killed xinhua,details school casualties nongovernmental inve...,5335


In [34]:
##
#train_df = df_train.reset_index(drop=True)
#test_df = df_test.reset_index(drop=True)

In [35]:
val_df = df[3800:4000].reset_index(drop=True) #added reset
val_df.head()

,question,context,answers
0,many game year awards twilight princess receive,time release twilight princess considered grea...,several
1,establishment politics court shift favor,death zhengde ascension jiajing politics court...,neoconfucian establishment
2,three ways antibiotics classified,antibacterial antibiotics commonly classified ...,mechanism action chemical structure spectrum a...
3,greenhouses solar energy,greenhouses convert solar light heat enabling ...,convert solar light heat
4,many homes rebuilt,executive vice governor wei hong confirmed nov...,200000


In [36]:
train_df.head()

,question,context,answers
0,publication alyssa rosenberg write,christopher orr writing atlantic also criticis...,washington post
1,frédéric trouble playing result growing illness,3 december chopin complained bad health incomp...,piano
2,name beyoncés first solo album,beyoncé giselle knowlescarter biːˈjɒnseɪ beeyo...,dangerously love
3,long birth child performed,january 7 2012 beyoncé gave birth first child ...,five months
4,many students killed xinhua,details school casualties nongovernmental inve...,5335


In [37]:
train_df.shape,  test_df.shape, val_df.shape

((3800, 3), (1000, 3), (200, 3))

# Create pytorch lightning datamodule

In [38]:
#LightningDataModule
class SquadDataModule(pl.LightningModule):
  def __init__(
      self,
      train_df: pd.DataFrame,
      test_df: pd.DataFrame,
      tokenizer:T5Tokenizer,
      batch_size: int = 8,
      source_max_token_len: int = 512,#396,
      target_max_token_len: int =  64#75,#32,
      ):
    super().__init__()
    self.train_df = train_df
    self.test_df = test_df
    self.tokenizer = tokenizer
    self.batch_size = batch_size
    self.source_max_token_len = source_max_token_len
    self.target_max_token_len = target_max_token_len

  def setup(self,stage=None):
    self.train_dataset = SquadQADataset(
        self.train_df,
        self.tokenizer,
        self.source_max_token_len,
        self.target_max_token_len
        )

    self.test_dataset = SquadQADataset(
    self.test_df,
    self.tokenizer,
    self.source_max_token_len,
    self.target_max_token_len
    )
 
  def train_dataloader(self):
    return DataLoader(
        self.train_dataset,
        batch_size=self.batch_size,
        shuffle=True,
        num_workers=2
        )
  def val_dataloader(self):
    return DataLoader(
        self.test_dataset,
        batch_size=self.batch_size,
        num_workers=2
        )

  def test_dataloader(self):
    return DataLoader(
        self.test_dataset,
        batch_size=1,
        num_workers=2  #was 4 before
        )

In [39]:
BATCH_SIZE = 2
N_EPOCHS = 5

data_module = SquadDataModule(train_df, test_df, tokenizer, batch_size=BATCH_SIZE)
data_module.setup()

## Loading and finetuning the T5-base model

## Building the PyTorch lightning module using T5ForConditionalGeneration model

In [40]:
class SquadQAModel(pl.LightningModule):
  def __init__(self):
    super().__init__()
    self.config = T5Config.from_pretrained(MODEL_NAME,dropout_rate =0.3,feed_forward_proj='relu',layer_norm_epsilon=1e-6)
    self.model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME,config=self.config,ignore_mismatched_sizes=True)#,return_dict=True)


  def forward(self, input_ids, attention_mask, labels=None):
    output = self.model(
        input_ids, 
        attention_mask=attention_mask,
        labels=labels)

    return output.loss, output.logits

  def training_step(self, batch, batch_idx):
    input_ids = batch['input_ids']
    attention_mask=batch['attention_mask']
    labels = batch['labels']
    loss, outputs = self(input_ids, attention_mask, labels)
    self.log("train_loss", loss, prog_bar=True, logger=True)
    return {"loss": loss, "predictions":outputs, "labels": labels}

  def validation_step(self, batch, batch_idx):
    input_ids = batch['input_ids']
    attention_mask=batch['attention_mask']
    labels = batch['labels']
    loss, outputs = self(input_ids, attention_mask, labels)
    self.log("val_loss", loss, prog_bar=True, logger=True)
    return loss

  def test_step(self, batch, batch_idx):
    input_ids = batch['input_ids']
    attention_mask=batch['attention_mask']
    labels = batch['labels']
    loss, outputs = self(input_ids, attention_mask, labels)
    self.log("test_loss", loss, prog_bar=True, logger=True)
    return loss

  def configure_optimizers(self):

    optimizer = AdamW(self.parameters(), lr=0.0003) #lr changed
    return optimizer

In [41]:
model = SquadQAModel() 

Downloading:   0%|          | 0.00/892M [00:00<?, ?B/s]

## Using trainer from pytorch lightning to finetune model using our dataset

In [42]:
# To record the best performing model using checkpoint

checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints",
    filename="best-checkpoint",
    save_top_k=1,
    verbose=True,
    monitor="val_loss",
    mode="min"
)

In [44]:
trainer = pl.Trainer(
    callbacks=[checkpoint_callback],
    max_epochs=N_EPOCHS,
    accelerator='gpu',
    devices=1
)

## Loading Tensorboard

In [46]:
trainer.fit(model, datamodule = data_module)

/opt/conda/lib/python3.7/site-packages/transformers/optimization.py:310: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  FutureWarning,


Sanity Checking: 0it [00:00, ?it/s]

/opt/conda/lib/python3.7/site-packages/torch/utils/data/dataloader.py:490: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  cpuset_checked))


Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [48]:
trainer.test(model, dataloaders=data_module.test_dataloader())  

Testing: 0it [00:00, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     1.395475149154663     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.395475149154663}]

# Predictions

In [67]:
trained_model = SquadQAModel.load_from_checkpoint("checkpoints/best-checkpoint.ckpt")
trained_model.freeze() # 

## Generate answers for the questions in the validation set

In [68]:
def generate_answer(question):
  source_encoding=tokenizer(
      question["question"],
      question['context'],
      max_length = 512,
      padding="max_length",
      truncation=True,
      return_attention_mask=True,
      add_special_tokens=True,
      return_tensors="pt"

  )

  generated_ids = trained_model.model.generate(
      input_ids=source_encoding["input_ids"],
      attention_mask=source_encoding["attention_mask"],
      num_beams=1,  # greedy search
      max_length=80,
      repetition_penalty=2.5,
      early_stopping=True,
      use_cache=True)
  
  preds = [
          tokenizer.decode(generated_id, skip_special_tokens=True, clean_up_tokenization_spaces=True)
          for generated_id in generated_ids
  ]

  return "".join(preds)

In [52]:
sample_question = val_df.iloc[20]

In [53]:
sample_question["question"]

'place game take nintendos official list 100 greatest nintendo games time'

In [54]:
sample_question["answers"]  

'16th'

In [55]:
generate_answer(sample_question) 

'4th'

In [56]:
sample_question = val_df.iloc[66]

In [57]:
sample_question["question"]

'beyonce said embodies definition inspiration strong woman'

In [58]:
sample_question["answers"]

'oprah winfrey'

In [59]:
generate_answer(sample_question)    

'oprah winfrey'

In [60]:
sample_question = val_df.iloc[114]

In [61]:
sample_question["question"]

'besides sprectrum activity chemical structure antibacterial antibiotics classified'

In [62]:
sample_question["answers"]

'mechanism action'

In [63]:
generate_answer(sample_question)

'mechanism action'